# 画像分類コンペ実践

転移学習・データ拡張・TTA・fold アンサンブルで画像分類コンペに取り組む流れを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/image-classification`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# timm パッケージの確認・自動インストールと、画像コンペ形式のメタデータ CSV および合成 RGB 画像生成関数を準備します。
import os
import subprocess
import sys

try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import numpy as np
import pandas as pd
import torch

def _make_synth_images(label_codes, seed=42):
    """各クラス固有の色・空間パッチ特徴を持つ 64x64 RGB 画像テンソルを生成します。"""
    rng = np.random.default_rng(seed)
    imgs = rng.normal(0.0, 0.25, size=(len(label_codes), 3, 64, 64)).astype(np.float32)
    palette = [
        (1.6, -0.8, -0.8),
        (-0.8, 1.6, -0.8),
        (-0.8, -0.8, 1.6),
        (1.3, 1.3, -0.9),
        (-0.9, 1.3, 1.3),
    ]
    # 水平反転（データ拡張）してもクラスの意味が壊れない左右対称の空間パッチを配置します
    patches = [(4, 22, 8, 56), (23, 41, 8, 56), (42, 60, 8, 56), (8, 56, 8, 56), (18, 46, 18, 46)]
    for idx, code in enumerate(label_codes):
        c = int(code) % len(palette)
        r0, r1, c0, c1 = patches[c]
        for ch in range(3):
            imgs[idx, ch, r0:r1, c0:c1] += palette[c][ch]
    return torch.tensor(imgs)

def _ensure_image_comp_data():
    data_dir = os.path.abspath("./.kaggle_input/image-classification")
    os.makedirs(data_dir, exist_ok=True)
    orig_read_csv = pd.read_csv
    if not getattr(orig_read_csv, "_is_img_comp_wrapped", False):
        def _patched_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/<competition>/"):
                if not os.path.exists(filepath_or_buffer):
                    rel = filepath_or_buffer[len("/kaggle/input/<competition>/"):]
                    filepath_or_buffer = os.path.join(data_dir, rel)
            return orig_read_csv(filepath_or_buffer, *args, **kwargs)
        _patched_read_csv._is_img_comp_wrapped = True
        pd.read_csv = _patched_read_csv

    train_path = os.path.join(data_dir, "train.csv")
    if os.path.exists(train_path):
        print(f"Using existing image competition metadata at: {train_path}")
        return

    rng = np.random.default_rng(42)
    # 5-fold StratifiedKFold で各クラス十分な件数を保ちつつ不均衡な分布を作成します（計 60 件）
    labels = np.array([0] * 18 + [1] * 14 + [2] * 12 + [3] * 9 + [4] * 7)
    rng.shuffle(labels)
    pd.DataFrame({
        "image_id": [f"img_{i:04d}.jpg" for i in range(len(labels))],
        "label": labels,
    }).to_csv(train_path, index=False)
    print(f"Image competition metadata ready at: {data_dir}")

_ensure_image_comp_data()

## 概要

画像分類コンペでは、ImageNet 等で事前学習済みのモデルを使う **転移学習** が標準です。
ゼロから学習するより少データ・短時間で高精度を狙えます。L7（CV）の実践編です。

> **Note:** 画像コンペは GPU が前提です。デバッグは少枚数・小解像度で行い、本学習だけ GPU を有効化して クォータを節約します（L7-09 参照）。

## 1. データ確認

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/<competition>/train.csv")  # image_id, label
df["label"].value_counts()   # クラス分布（不均衡を確認）

クラスの偏り・画像サイズ・破損画像を確認し、層化した 交差検証 を設計します。

## 2. 事前学習モデルのファインチューニング

In [ ]:
# クラス数を訓練メタデータから取得します
n_classes = int(df["label"].nunique())

In [ ]:
import timm, torch.nn as nn

model = timm.create_model("tf_efficientnet_b0", pretrained=True, num_classes=n_classes)
# 通常の PyTorch 学習ループ（AdamW + 小さめ学習率 + 早期終了）

事前学習時より**小さい学習率**で微調整し、検証で早期終了します。

## 3. データ拡張と TTA

In [ ]:
# 学習: 反転・回転・切り抜き・色変換などで水増し（過学習抑制）
# 推論: TTA（複数の拡張版を予測して平均）でスコア底上げ

データ拡張 は画像コンペの効きどころです。正規化の統計量は学習・検証で必ずそろえます。

## 4. fold アンサンブル

各 fold のモデルでテストを予測し、平均します。さらに別アーキテクチャを混ぜると多様性が出ます。

In [ ]:
# StratifiedKFold (5-fold) で fold ごとに独立したモデルをファインチューニングし、別モデルとして保持します
import copy
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
y_codes, _ = pd.factorize(df["label"], sort=True)
X_images = _make_synth_images(y_codes, seed=42)
y_tensor = torch.tensor(y_codes, dtype=torch.long)

# 学習前の ImageNet 事前学習済み重みをテンプレートとして保持（fold 間の情報漏洩を防止）
pretrained_template = copy.deepcopy(model)

class _FoldPredictor:
    def __init__(self, net, dev):
        self.net = net
        self.dev = dev
    def predict(self, batch_tensor):
        self.net.eval()
        with torch.no_grad():
            logits = self.net(batch_tensor.to(self.dev))
            return torch.softmax(logits, dim=-1).cpu().numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_models = []
fold_val_losses = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_codes)), y_codes)):
    torch.manual_seed(42 + fold)
    fold_net = copy.deepcopy(pretrained_template).to(device)
    fold_net.reset_classifier(n_classes)
    for param_name, param in fold_net.named_parameters():
        param.requires_grad = param_name.startswith("classifier")

    optimizer = torch.optim.AdamW(fold_net.classifier.parameters(), lr=8e-3, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss()
    tr_loader = DataLoader(TensorDataset(X_images[tr_idx], y_tensor[tr_idx]), batch_size=24, shuffle=True)

    best_loss = float("inf")
    best_state = None
    for _epoch in range(4):
        fold_net.eval()
        fold_net.classifier.train()
        for xb, yb in tr_loader:
            # 水平反転による簡易データ拡張
            if torch.rand(1).item() < 0.5:
                xb = torch.flip(xb, dims=[-1])
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(fold_net(xb), yb)
            loss.backward()
            optimizer.step()

        fold_net.eval()
        with torch.no_grad():
            val_loss = criterion(fold_net(X_images[va_idx].to(device)), y_tensor[va_idx].to(device)).item()
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in fold_net.state_dict().items()}

    fold_net.load_state_dict(best_state)
    fold_val_losses.append(best_loss)
    fold_models.append(_FoldPredictor(fold_net, device))

test_true_labels = np.arange(min(n_classes, 5))
test = _make_synth_images(test_true_labels, seed=999)

In [ ]:
test_pred = sum(fold_models[k].predict(test) for k in range(5)) / 5

In [ ]:
fold0_pred = fold_models[0].predict(test)
fold1_pred = fold_models[1].predict(test)
print("Mean fold best val loss:", round(float(np.mean(fold_val_losses)), 4))
print("Max prediction diff (fold 0 vs fold 1):", round(float(np.abs(fold0_pred - fold1_pred).max()), 4))
print("Fold-averaged test_pred shape:", test_pred.shape)
print("Predicted classes:", test_pred.argmax(axis=1), "| True classes:", test_true_labels)

> **Warning:** 検証時の前処理・正規化を推論と一致させること。ここがずれると 交差検証 が当てにならず、 過学習やシェイクアップの原因になります。

## 関連トラック

- L7-04 CV I：CNN 基礎
- L7-05 CV II：転移学習・ファインチューニング
- L7-09 GPU/TPU・メモリ最適化